In [49]:
import numpy as np
import pandas as pd 
import json
import re
import matplotlib.pyplot as plt
import umap
from pathlib import Path
from sklearn.cluster import KMeans
from solution_utils import build_tables, get_users_text, vectorize_topic_texts, SEED, find_duplicate_groups, read_dialogs, get_groups
from sklearn.metrics import homogeneity_score, completeness_score, silhouette_score, adjusted_rand_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components

In [22]:
data = pd.read_json('hw1_train.jsonl', lines=True)

data

,dialog_id,language,n_user_turns,turns,labels
0,d094012177e,en,7,"[{'role': 'user', 'text': 'it’s like: 1+1=2, b...","{'task_type': 'analysis', 'reaction': 'positiv..."
1,d306fbe61fe,en,8,"[{'role': 'user', 'text': 'Can you help me tra...","{'task_type': 'information_seeking', 'reaction..."
2,dd6905952c7,en,5,"[{'role': 'user', 'text': 'ask questions about...","{'task_type': 'creation', 'reaction': 'neutral..."
3,d767a675896,en,6,"[{'role': 'user', 'text': 'Calculate each prom...","{'task_type': 'information_seeking', 'reaction..."
4,d7a6280ce8e,en,8,"[{'role': 'user', 'text': 'mass spectrometer, ...","{'task_type': 'information_seeking', 'reaction..."
...,...,...,...,...,...
9995,d745e2d0b74,en,8,"[{'role': 'user', 'text': 'which countries off...",None
9996,d2248b50e1e,en,6,"[{'role': 'user', 'text': 'Medication safety, ...",None
9997,dba01d56b0d,en,10,"[{'role': 'user', 'text': 'Your instructions a...",None
9998,d07a2ffacb8,en,5,"[{'role': 'user', 'text': 'write a story for p...",None


In [23]:
print(f'доля записей с labels {data['labels'].notna().mean()}')
data = data.drop(columns='labels')

доля записей с labels 0.3


ответы нейронки смотреть смысла нет, так как пользователь задаёт тему и ведет её, нейронка только поддакивает

In [24]:
dialogs, turn = build_tables(data.to_dict('records'))

user = dialogs.iloc[0, 2]

# for replica in user:
#     print(replica)

print (user[0])

dialogs

i


,language,n_user_turns,user_text,assistant_text,user_len,assistant_len,reaction,task_type
dialog_id,,,,,,,,
d094012177e,en,7,"it’s like: 1+1=2, but 1+1=2+1+1=? is undefined...",Introducing scientific terms from astrophysics...,8453,10492,None,None
d306fbe61fe,en,8,Can you help me train a small language model t...,Creating a small language model from scratch i...,1278,32255,None,None
dd6905952c7,en,5,ask questions about a fictional voice actor i ...,1. How did this fictional voice actor get into...,3326,11938,None,None
d767a675896,en,6,Calculate each prompt based off of data and sh...,"To calculate Species Richness, simply count th...",1115,9253,None,None
d7a6280ce8e,en,8,"mass spectrometer, the length of AC rod will i...",As an AI language model I don't have enough in...,374,7528,None,None
...,...,...,...,...,...,...,...,...
d745e2d0b74,en,8,which countries offer special programs to enco...,There are several countries that offer special...,433,11913,None,None
d2248b50e1e,en,6,"Medication safety, \n\nMedication administrati...",This documentation is important in ensuring co...,2776,5259,None,None
dba01d56b0d,en,10,Your instructions are: For the following quest...,"As an AI language model, I do not possess the ...",995,8526,None,None


In [25]:
user_text = get_users_text(data.to_dict('records'), text_type='user')

key_words_classterize = [] # будет хранить диалоги по которым будем кластеризовать
for payload in user_text['text']:
    key_words_classterize.append(payload[:2])

key_words_classterize

[['it’s like: 1+1=2, but 1+1=2+1+1=? is undefined. <–can you tell which modulus we can possibly use in this concept and into which it may result?\nIn the context of the equation “1+1=2+1+1=?,” if we want to explore the possibility of using a modulus to determine the result, we can consider different modulus values and see how they affect the equation.\n\nModulus, also known as the remainder when dividing a number by another, is often denoted by the symbol “%”. It can be used to find the remainder of a division operation.\n\nLet’s consider a few examples:\n\n1. Modulus 3:\nUsing modulus 3, we would evaluate remainders when dividing numbers by 3. Therefore:\n1 + 1 = 2 % 3 + 1 % 3 + 1 % 3\n= 2 + 1 + 1\n= 4\n\n2. Modulus 4:\nUsing modulus 4, we would evaluate remainders when dividing numbers by 4. Therefore:\n1 + 1 = 2 % 4 + 1 % 4 + 1 % 4\n= 2 + 1 + 1\n= 4\n\n3. Modulus 5:\nUsing modulus 5, we would evaluate remainders when dividing numbers by 5. Therefore:\n1 + 1 = 2 % 5 + 1 % 5 + 1 % 5\n

# мой план:
1) сделать векторизацию tf-idf 
2) подбирать кластеры kmeans++
3)  сравнивать метрики (силуэт и тд)
4) назвать темы 
5) проверть при более полном наборе диалогов
6) утвердить окончательно
7) сделать тествоую выборку (разметить темы самому)
8) обучать, сравнивать, деплоить

In [26]:
# X,  tf_idfer = vectorize_topic_texts(key_words_classterize) # наши диалоги в циферках


In [27]:
# reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=SEED)

# X_umap = reducer.fit_transform(X)

# plt.figure(figsize=(10, 7))
# plt.scatter(X_umap[:, 0], X_umap[:, 1], s=10, alpha=0.7)

# plt.title('UMAP visualization raw data')
# plt.xlabel('UMAP 1')
# plt.ylabel('UMAP 2')
# plt.show()

In [28]:
# empty_mask = np.asarray(X.getnnz(axis=1) == 0).ravel()
# print("Диалогов без TF-IDF-признаков:", empty_mask.sum())

а что это за диалоги?

In [29]:
# empty_indices = np.where(empty_mask)[0]

# for i in empty_indices[:20]:
#     print(f"Индекс: {i}")
#     print(repr(key_words_classterize[i]))
#     print("---")

In [30]:
# scores_silhuet = []
# for k in range (3, 15):
#     kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=SEED)
#     cluster_labels = kmeans.fit_predict(X)

#     scores_silhuet.append((k, silhouette_score(X, cluster_labels, random_state=SEED, sample_size=None, metric='euclidean')))

# score_table = pd.DataFrame(scores_silhuet, columns=["k", "silhouette_score"])
# print(score_table)

In [31]:
# plt.figure(figsize=(8, 5))
# plt.plot(score_table["k"], score_table["silhouette_score"], marker="o")

# plt.xlabel("Количество кластеров (k)")
# plt.ylabel("Silhouette Score (Евклидово расстояние)")
# plt.title("Выбор количества кластеров")
# plt.xticks(score_table["k"])
# plt.grid(True)
# plt.show()

мда, силуэт порядка 0,001 это конечно сильно. можно попробовать глянуть косинусное расстояние для сравнения, а потом переходить к рассмотрению полных диалогов user и может быть даже к user+assistant

In [32]:
# scores_silhuet_cos = []
# for k in range (3, 15):
#     kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=SEED)
#     cluster_labels_cos = kmeans.fit_predict(X)

#     scores_silhuet_cos.append((k, silhouette_score(X, cluster_labels_cos, random_state=SEED, sample_size=None, metric='cosine')))

# score_table_cos = pd.DataFrame(scores_silhuet_cos, columns=["k", "silhouette_score"])

In [33]:
# plt.figure(figsize=(8, 5))
# plt.plot(score_table_cos["k"], score_table_cos["silhouette_score"], marker="o")

# plt.xlabel("Количество кластеров (k)")
# plt.ylabel("Silhouette Score (Косинусное расстояние)")
# plt.title("Выбор количества кластеров")
# plt.xticks(score_table_cos["k"])
# plt.grid(True)
# plt.show()

в целом этого следовало ожидать, не, ну а вдруг) 
переходим к обучению на всех текстах user'a

In [34]:
# key_words_classterize_all_user = [] # будет хранить диалоги по которым будем кластеризовать
# for payload in user_text['text']:
#     key_words_classterize_all_user.append(payload)

# X_all_user,  tf_idfer_all_user = vectorize_topic_texts(key_words_classterize_all_user) # наши диалоги в циферках

# empty_mask = np.asarray(X_all_user.getnnz(axis=1) == 0).ravel()
# print("Диалогов без TF-IDF-признаков:", empty_mask.sum())

# scores_silhuet_all_user = []
# for k in range (3, 15):
#     kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=SEED)
#     cluster_labels_all_user = kmeans.fit_predict(X)

#     scores_silhuet_all_user.append((k, silhouette_score(X, cluster_labels_all_user, random_state=SEED, sample_size=None, metric='euclidean')))

# score_table_all_user = pd.DataFrame(scores_silhuet_all_user, columns=["k", "silhouette_score"])
# print(score_table_all_user)

In [35]:
# plt.figure(figsize=(8, 5))
# plt.plot(score_table_all_user["k"], score_table_all_user["silhouette_score"], marker="o")

# plt.xlabel("Количество кластеров (k)")
# plt.ylabel("Silhouette Score")
# plt.xticks(score_table_all_user["k"])
# plt.grid(True)
# plt.title('Полные текста user')
# plt.show()

попробуем еще 1 тест: урежем normalize и попробуем с менее ограниченными словами посмотреть

In [36]:
# X,  tf_idfer = vectorize_topic_texts(key_words_classterize, full_normalize=False) # наши диалоги в циферках

# empty_mask = np.asarray(X.getnnz(axis=1) == 0).ravel()
# print("Диалогов без TF-IDF-признаков:", empty_mask.sum())

# scores_silhuet = []
# for k in range (3, 15):
#     kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=SEED)
#     cluster_labels = kmeans.fit_predict(X)

#     scores_silhuet.append((k, silhouette_score(X, cluster_labels, random_state=SEED, sample_size=None, metric='euclidean')))

# score_table = pd.DataFrame(scores_silhuet, columns=["k", "silhouette_score"])
# print(score_table)



In [37]:
# plt.figure(figsize=(8, 5))
# plt.plot(score_table["k"], score_table["silhouette_score"], marker="o")

# plt.xlabel("количетсво класстеров")
# plt.ylabel("Silhouette Score (Евклидово расстояние)")
# plt.title("Другой normalize")
# plt.xticks(score_table["k"])
# plt.grid(True)
# plt.show()

## Анализ диалогов

In [38]:
# 0 - chat gpt (есть в мире дураки (выбросы))
# 1 - games (иногда заигрывания 😏)
# 2 - программирование (очень хорошо распределил)
# 3 - просьбы что-то сделать? (упомянается конкретное действие)
# 4 - просьба что-то расчитать, прикинуть (что-то люто непонятное)
# 5 - что-то для школьников/простое (возможно выбросы)
# 6 - Упомянаются какие-то люди
# 7 - хрень не на русском (обычно что-то перевести или реально запросы)

In [39]:
kmeans = KMeans(n_clusters=8, init='k-means++', n_init=10, max_iter=300, random_state=SEED)
cluster_labels = kmeans.fit_predict(X)

NameError: name 'X' is not defined

In [ ]:
# dialog_table_to_analyze = []
# for theme_idx in range(8):
#     indexes = cluster_labels == theme_idx
#     themed_dialog = data[indexes]

#     print(f'Кластер {theme_idx}, количество диалогов: {len(themed_dialog)}')

#     n_samples = min(15, len(themed_dialog))

#     if n_samples > 0:
#         selected_indices = np.random.choice(
#             len(themed_dialog),
#             size=n_samples,
#             replace=False
#         )
#         get_dialog = themed_dialog.iloc[selected_indices].values.tolist()
#     else:
#         get_dialog = []

#     dialog_table_to_analyze.append((theme_idx, get_dialog))

# dialog_table_to_analyze = pd.DataFrame(dialog_table_to_analyze, columns=['theme', 'dialog'])

Кластер 0, количество диалогов: 3718
Кластер 1, количество диалогов: 1390
Кластер 2, количество диалогов: 820
Кластер 3, количество диалогов: 460
Кластер 4, количество диалогов: 1907
Кластер 5, количество диалогов: 1265
Кластер 6, количество диалогов: 358
Кластер 7, количество диалогов: 82


In [ ]:
# dialog_table_to_analyze['dialog'][0]

[['d053462d8ec',
  'en',
  6,
  [{'role': 'user',
    'text': 'Wrote m/m romance about a strong warrior and shy forest elf with long golden hair. Add dialogue and describe how they meet, thank you'},
   {'role': 'assistant',
    'text': 'Title: "Whispered Secrets"\n\n---\n\nIn the heart of the enchanted forest, the air was thick with tranquility as rays of sunlight filtered through the lush foliage. Amidst the dancing foliage stood a towering warrior named Brant, his presence exuding strength and determination. His muscular build was a testament to endless hours of training, while his piercing green eyes held a hint of vulnerability. Brant\'s armor clanked softly as he navigated the forest, his sword strapped securely to his side.\n\nUnbeknownst to Brant, nestled within the vibrant greenery, was a shy forest elf named Eirik. His porcelain skin glowed ethereally, camouflaging him amongst the flora. Eirik\'s nervousness was betrayed by the way his hands trembled and his gaze darted aroun

для кластера 4 имеем явно 1 человека который фанфики пишет. сохраним id его диалогов

In [ ]:
# funfic_id = data[cluster_labels == 4]['dialog_id'].tolist()
# funfic_id

['d094012177e',
 'dd6905952c7',
 'd60e2d33bc9',
 'd37ac19fd7a',
 'dd54ca8089d',
 'd5489837923',
 'dd8832f4141',
 'd4c5e547fb1',
 'd913b9435ee',
 'de1ec606ff3',
 'db958cd1f31',
 'd7a4a7893a6',
 'd59ec97182e',
 'd1666f4cf5e',
 'd06bfb34ef5',
 'd6d4150432e',
 'd6d5d596a7f',
 'd1de772cfdf',
 'da9199e1a01',
 'd82cd44c9a3',
 'dbf83570094',
 'd76c341656d',
 'ddd3ec665a0',
 'd7a6e619cdc',
 'd394ee15433',
 'de4e0bc2124',
 'd7f71473eff',
 'd4157de1bfd',
 'dcb5714daa2',
 'df01ba1b682',
 'd01a4ddc371',
 'd42d15b8cad',
 'dbc4865f03c',
 'da749f23679',
 'dd21fc182de',
 'd98871102ba',
 'd99f3b978c6',
 'dd54dd06765',
 'db80fc930f3',
 'd800498312b',
 'd7784a1987b',
 'd61dbf1cdae',
 'd013bfd857c',
 'd2e6bf515b8',
 'd4db79aaf58',
 'db3a30e7e53',
 'd03abc548ea',
 'd192fc817e9',
 'dae044ad0a1',
 'd79948426c3',
 'd2fa7cce94d',
 'daf6a656a3f',
 'd0ea704fba7',
 'deb6bedaf62',
 'd573f6fb73d',
 'd84c271e1bd',
 'df4c5377709',
 'da68c893213',
 'd78fabefc1a',
 'df290e1a2ae',
 'dbf4f4bfda1',
 'd7b12128f62',
 'd08d3c

In [ ]:
# dist = kmeans.transform(X)               # (n_dialogs, 8)
# labels = kmeans.labels_                  # или ваш cluster_labels

# own_dist = dist[np.arange(X.shape[0]), labels]   # расстояние до своего центроида

# # margin: насколько диалог ближе к своему центру, чем ко второму по близости
# sorted_dist = np.sort(dist, axis=1)
# margin = sorted_dist[:, 1] - sorted_dist[:, 0]   # маленький = на границе двух кластеров

# N = 10
# rows = []
# for k in range(14):
#     idx = np.where(labels == k)[0]

#     center_idx = idx[np.argsort(own_dist[idx])[:N]]          # самые типичные
#     far_idx    = idx[np.argsort(-own_dist[idx])[:N]]         # самые далёкие от центра
#     border_idx = idx[np.argsort(margin[idx])[:N]]            # на стыке с другим кластером
#     random_idx = np.random.choice(idx, min(N, len(idx)), replace=False)

#     # к какому кластеру "тянет" пограничные диалоги
#     neighbor = np.argsort(dist[border_idx], axis=1)[:, 1]

#     for kind, ids in [('center', center_idx), ('far', far_idx),
#                       ('border', border_idx), ('random', random_idx)]:
#         for i in ids:
#             rows.append({
#                 'theme': k,
#                 'kind': kind,
#                 'second_cluster': np.argsort(dist[i])[1],
#                 'dist': own_dist[i],
#                 'dialog': data.iloc[i],
#             })

# df = pd.DataFrame(rows)
# # print(df[(df.theme == 0) & (df.kind == 'center')]['dialog'].tolist())
# df

,theme,kind,second_cluster,dist,dialog
0,0,center,1,0.117141,dialog_id ...
1,0,center,1,0.117141,dialog_id ...
2,0,center,1,0.117141,dialog_id ...
3,0,center,1,0.117141,dialog_id ...
4,0,center,1,0.117141,dialog_id ...
...,...,...,...,...,...
315,7,random,2,0.847298,dialog_id ...
316,7,random,2,0.838333,dialog_id ...
317,7,random,2,0.787697,dialog_id ...
318,7,random,2,0.890549,dialog_id ...


In [ ]:
# df[(df['theme'] == 1) & (df['kind'] == 'center')]['dialog'].tolist()[7]['turns']

[{'role': 'user',
  'text': 'I have a code in colab that is being run as !python -i 0   now instead of -i 0 I want to run it for all values between 0 and 512. How can I do that'},
 {'role': 'assistant',
  'text': 'You can use a loop to iterate over all the values between 0 and 512 and run the code for each value. Here\'s an example:\n\n```\nfor i in range(513):\n  !python -i {i}\n```\n\nThis will run the code with "-i" parameter set to each value between 0 and 512. Note that this will run the code 513 times, once for each value from 0 to 512 inclusive.'},
 {'role': 'user',
  'text': 'do you think it will work ? does adding {i} read the value ?'},
 {'role': 'assistant',
  'text': 'Yes, adding `{i}` inside the command string `"!python -i {i}"` will substitute the value of the variable `i` during each iteration of the loop. So the loop will run `!python -i 0`, then `!python -i 1`, then `!python -i 2`, and so on until it reaches `!python -i 512`. \n\nAssuming the original command line `!py

# Итоговый список тем:
0) IT (программирование) (coding)
1) вопросы про ИИ 
2) перевод на другие языки (работа с языками)
3) работа с текстом
4) объяснение/разъяснение общих вопросов (questioning/explanation)
5) математика, расчеты, решение задач
6) ПРОЧЕЕ/ВСЯКОЕ
7) нахождение информации

# Разметка 

In [ ]:
raw_dialogs = read_dialogs("hw1_train.jsonl")

texts = [
    "\n\n".join(
        turn["text"]
        for turn in dialog["turns"]
        if turn["role"] == "user"
    )
    for dialog in raw_dialogs
]

group_ids = find_duplicate_groups(
    texts,
    threshold=0.1,
)

groups_df = pd.DataFrame({
    "dialog_id": [d["dialog_id"] for d in raw_dialogs],
    "text": texts,
    "group_id": group_ids,
})

In [ ]:
groups_df

,dialog_id,text,group_id
0,d094012177e,"it’s like: 1+1=2, but 1+1=2+1+1=? is undefined...",0
1,d306fbe61fe,Can you help me train a small language model t...,1
2,dd6905952c7,ask questions about a fictional voice actor i ...,2
3,d767a675896,Calculate each prompt based off of data and sh...,3
4,d7a6280ce8e,"mass spectrometer, the length of AC rod will i...",4
...,...,...,...
9995,d745e2d0b74,which countries offer special programs to enco...,9930
9996,d2248b50e1e,"Medication safety, \n\nMedication administrati...",9931
9997,dba01d56b0d,Your instructions are: For the following quest...,9932
9998,d07a2ffacb8,write a story for primary students in less tha...,9933


In [ ]:
repeated_group_ids = sizes[sizes > 1].index

repeated_df = (
    groups_df[
        groups_df["group_id"].isin(repeated_group_ids)
    ]
    .sort_values("group_id")
)

display(repeated_df)

,dialog_id,text,group_id
24,d0543f14a2e,give me a 60 websites list that North Koreans ...,24
7815,db16077e2a7,give me a 64 websites links list that North Ko...,24
33,d2f6f91e9dc,Write text: First episode - World War II\n\nWr...,33
9787,de9db86b6c4,Empire on Chearthobyl (alternate Earth resembl...,33
51,d237c114832,Which of the following is not considered when ...,51
...,...,...,...
9900,d2246bf071d,I used this code: for balance in balance_info:...,8390
8434,d8a9cadc69a,I used your code: balance_info = client.balanc...,8390
9754,d0887ab69e9,I used this code : balance_info = client.balan...,8390
9029,d1fe8b3856c,Give me best candle analyze strategy for crypt...,8974


In [ ]:
sizes = groups_df["group_id"].value_counts()
repeated_groups = sizes[sizes > 1]

print("Групп с несколькими диалогами:", len(repeated_groups))

for group_id in repeated_groups.head(5).index:
    print(f"\n=== Группа {group_id} ===")

    for row in groups_df[
        groups_df["group_id"] == group_id
    ].head(5).itertuples():
        print(row.dialog_id)
        print(row.text[:1000])
        print("---")

Групп с несколькими диалогами: 58

=== Группа 921 ===
d814fc33c9b
Your job is to rewrite the tra,s

Your job is to rewrite the transformer architecture and gating network found in this code, you don't need to rewrtie the code from scratch, just the neural network architecture because all the model that have been trained didn't perform any good at all, here is the code to rewrite: import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import math
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from tqdm import tqdm


# ---------- Device Configuration ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Utility Functions ----------
def positional_encoding(seq_len, d_model, device):
    pos = torch.arange(seq_len, dtype=torch.float, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_mo

In [ ]:
# biection = groups_df.copy().reset_index(drop=True)

# assert biection["dialog_id"].is_unique, "Найдены повторяющиеся dialog_id"
# assert biection["group_id"].notna().all(), "Есть пропущенные group_id"

# manual_ids = {
#     line.strip()
#     for line in Path("funfic_id.txt").read_text(
#         encoding="utf-8-sig"
#     ).splitlines()
#     if line.strip()
# }

# if not manual_ids:
#     raise ValueError("Файл 1.txt пуст")

# unknown_ids = manual_ids - set(biection["dialog_id"])

# if unknown_ids:
#     raise ValueError(
#         "Некоторые ID из 1.txt отсутствуют в groups_df: "
#         f"{sorted(unknown_ids)[:10]}. "
#         "Проверь файл и убедись, что используешь полный groups_df."
#     )

# biection["group_id"] = pd.factorize(
#     biection["group_id"], sort=False
# )[0]

# manual_mask = biection["dialog_id"].isin(manual_ids)

# touched_groups = biection.loc[
#     manual_mask, "group_id"
# ].unique()

# merge_mask = biection["group_id"].isin(touched_groups)

# new_group_id = int(biection["group_id"].max()) + 1
# biection.loc[merge_mask, "group_id"] = new_group_id

# biection["group_id"] = pd.factorize(
#     biection["group_id"], sort=False
# )[0]

# assert len(biection) == len(groups_df)
# assert biection.loc[manual_mask, "group_id"].nunique() == 1

# print("Всего диалогов:", len(biection))
# print("Всего групп:", biection["group_id"].nunique())
# print("ID в ручном списке:", len(manual_ids))
# print("Диалогов в объединённой группе:", int(merge_mask.sum()))

Всего диалогов: 10000
Всего групп: 9881
ID в ручном списке: 55
Диалогов в объединённой группе: 55


In [ ]:
# split_test = GroupShuffleSplit(
#     n_splits=1,
#     test_size=0.15,
#     random_state=42,
# )

# dev_idx, test_idx = next(
#     split_test.split(biection, groups=biection["group_id"])
# )

# dev = biection.iloc[dev_idx].copy()
# test = biection.iloc[test_idx].copy()

# split_val = GroupShuffleSplit(
#     n_splits=1,
#     test_size=0.15 / 0.85,
#     random_state=43,
# )

# train_idx, val_idx = next(
#     split_val.split(dev, groups=dev["group_id"])
# )

# train = dev.iloc[train_idx].copy()
# val = dev.iloc[val_idx].copy()

# train["split"] = "train"
# val["split"] = "val"
# test["split"] = "test"

# split_biection = pd.concat([train, val, test], ignore_index=True)

# split_biection[["dialog_id", "group_id", "split"]].to_csv(
#     "topic_split.csv",
#     index=False,
# )

# print(split_biection["split"].value_counts())

split
train    7015
val      1493
test     1492
Name: count, dtype: int64


In [ ]:
all_text, _ = build_tables(data.to_dict('records'))

all_text.loc[test['dialog_id']]

,language,n_user_turns,user_text,assistant_text,user_len,assistant_len,reaction,task_type
dialog_id,,,,,,,,
d094012177e,en,7,"it’s like: 1+1=2, but 1+1=2+1+1=? is undefined...",Introducing scientific terms from astrophysics...,8453,10492,None,None
d767a675896,en,6,Calculate each prompt based off of data and sh...,"To calculate Species Richness, simply count th...",1115,9253,None,None
d61b201ed5c,en,10,A chemical supply company sells an acid soluti...,Let's assume the worker added x liters of deio...,3991,9411,None,None
db6c908c5a0,en,5,Show me a model from inception to full executi...,"As an AI language model, I cannot provide fina...",1259,5955,None,None
d530f30bf07,en,7,how to set service health preferences\nhow to ...,"To set service health preferences, you can fol...",248,9576,None,None
...,...,...,...,...,...,...,...,...
d663de70d7a,en,6,Write the declaration of independence\nNo i wa...,The Declaration of Independence\n\nIn Congress...,325,15722,None,None
d70dfe9ca11,en,10,All Hotel DOMINICAN REPUBLIC\nDetails for each...,1. Hard Rock Hotel & Casino Punta Cana\n2. Cas...,1012,16029,None,None
db5b533665f,en,7,Do that storyline please…but add with the dial...,In the desolated remains of what was once Goth...,21109,25898,None,None


In [ ]:
# all_text.loc['d8919400430']['user_text']

'naruto all characters reacted to https://www.fanfiction.net/s/11504497/1/Sonic-and-Lilac-Tides-of-Chaos\nnaruto all characters reacted to https://www.fanfiction.net/s/12075220/1/Sonic-Sash-Tides-of-Chaos-2-Bad-Bloodlines\nnaruto all characters reacted to https://www.fanfiction.net/s/12820555/1/Sonic-Sash-Tides-of-Chaos-3-Phantom-Forces\nnaruto all characters reacted to naruto in https://www.fanfiction.net/s/11504497/1/Sonic-and-Lilac-Tides-of-Chaos style\nnaruto in https://www.fanfiction.net/s/11504497/1/Sonic-and-Lilac-Tides-of-Chaos style'

### Используем kmeans

In [ ]:
# mask_train = data['dialog_id'].isin(train['dialog_id'])

In [ ]:
# user_text_train = get_users_text(data[mask_train].to_dict('records'), text_type='user')

# key_words_classterize_train = [] # будет хранить диалоги по которым будем кластеризовать
# for payload in user_text['text']:
#     key_words_classterize_train.append(payload)

In [ ]:
# X,  tf_idfer = vectorize_topic_texts(key_words_classterize_train, full_normalize=False) # наши диалоги в циферках

# kmeans = KMeans(n_clusters=8, init='k-means++', n_init=10, max_iter=300, random_state=SEED)
# cluster_labels = kmeans.fit_predict(X)

# dialog_table_to_analyze = []
# for theme_idx in range(8):
#     indexes = cluster_labels == theme_idx
#     themed_dialog = data[indexes]

#     print(f'Кластер {theme_idx}, количество диалогов: {len(themed_dialog)}')

#     n_samples = min(15, len(themed_dialog))

#     if n_samples > 0:
#         selected_indices = np.random.choice(
#             len(themed_dialog),
#             size=n_samples,
#             replace=False
#         )
#         get_dialog = themed_dialog.iloc[selected_indices].values.tolist()
#     else:
#         get_dialog = []

#     dialog_table_to_analyze.append((theme_idx, get_dialog))

# dialog_table_to_analyze = pd.DataFrame(dialog_table_to_analyze, columns=['theme', 'dialog'])

Кластер 0, количество диалогов: 2255
Кластер 1, количество диалогов: 1308
Кластер 2, количество диалогов: 1108
Кластер 3, количество диалогов: 1390
Кластер 4, количество диалогов: 1952
Кластер 5, количество диалогов: 140
Кластер 6, количество диалогов: 643
Кластер 7, количество диалогов: 1204


# Ручная разметка

In [ ]:
# ============================================================
# 1. Настройки
# ============================================================

DATA_PATH = "hw1_train.jsonl"
MANUAL_GROUP_PATH = "funfic_id.txt"  # None — если ручной группы нет

N_TRAIN = 933
N_VAL = 200
N_TEST = 200
N_TOTAL = N_TRAIN + N_VAL + N_TEST  # 1333


# ============================================================
# 2. Единственная новая функция:
#    отбор целых групп на точное количество диалогов
# ============================================================

def take_exact_groups(frame, target_size, rng):
    """Возвращает выбранные строки и остаток, не разрезая группы."""
    sizes = frame.groupby("group_id", sort=False).size()
    group_ids = sizes.index.to_numpy()
    group_sizes = sizes.to_numpy()

    # reachable[s] — можно ли набрать ровно s строк.
    reachable = np.zeros(target_size + 1, dtype=bool)
    reachable[0] = True

    previous_sum = np.full(target_size + 1, -1, dtype=int)
    previous_group = np.full(target_size + 1, -1, dtype=int)

    for position in rng.permutation(len(group_ids)):
        size = int(group_sizes[position])

        if size > target_size:
            continue

        # Копия нужна, чтобы одну группу не использовать повторно.
        old_reachable = reachable.copy()

        new_sums = (
            np.flatnonzero(
                old_reachable[:target_size + 1 - size]
            )
            + size
        )
        new_sums = new_sums[~reachable[new_sums]]

        reachable[new_sums] = True
        previous_sum[new_sums] = new_sums - size
        previous_group[new_sums] = position

        if reachable[target_size]:
            break

    if not reachable[target_size]:
        raise ValueError(
            f"Нельзя набрать ровно {target_size} строк "
            "из доступных целых групп."
        )

    selected_groups = []
    current_sum = target_size

    while current_sum > 0:
        position = previous_group[current_sum]
        selected_groups.append(group_ids[position])
        current_sum = previous_sum[current_sum]

    mask = frame["group_id"].isin(selected_groups)

    return frame.loc[mask].copy(), frame.loc[~mask].copy()


# ============================================================
# 3. Читаем данные и используем готовый get_groups
# ============================================================

raw_dialogs = read_dialogs(DATA_PATH)

texts = [
    "\n\n".join(
        turn["text"]
        for turn in dialog["turns"]
        if turn["role"] == "user"
    )
    for dialog in raw_dialogs
]

n_groups, group_ids = get_groups(
    texts,
    threshold=0.1,
)

groups_df = pd.DataFrame({
    "dialog_id": [d["dialog_id"] for d in raw_dialogs],
    "turns": [d["turns"] for d in raw_dialogs],
    "text": texts,
    "group_id": group_ids,
})

biection = groups_df.copy()

assert biection["dialog_id"].is_unique, "Повторяющиеся dialog_id"
assert biection["group_id"].notna().all(), "Пропущенные group_id"
assert len(biection) >= N_TOTAL, "Недостаточно диалогов"


# ============================================================
# 4. Объединяем ручную группу и связанные с ней группы
# ============================================================

if MANUAL_GROUP_PATH is not None:
    manual_ids = {
        line.strip()
        for line in Path(MANUAL_GROUP_PATH).read_text(
            encoding="utf-8-sig"
        ).splitlines()
        if line.strip()
    }

    if not manual_ids:
        raise ValueError(f"Файл {MANUAL_GROUP_PATH} пуст")

    unknown_ids = manual_ids - set(biection["dialog_id"])

    if unknown_ids:
        raise ValueError(
            f"ID из {MANUAL_GROUP_PATH} отсутствуют в данных: "
            f"{sorted(unknown_ids)[:10]}"
        )

    manual_mask = biection["dialog_id"].isin(manual_ids)

    touched_groups = biection.loc[
        manual_mask, "group_id"
    ].unique()

    merge_mask = biection["group_id"].isin(touched_groups)

    biection.loc[merge_mask, "group_id"] = (
        int(biection["group_id"].max()) + 1
    )

    print("ID в ручном списке:", len(manual_ids))
    print("Диалогов в объединённой группе:", int(merge_mask.sum()))

biection["group_id"] = pd.factorize(
    biection["group_id"],
    sort=False,
)[0]

full_group_sizes = biection.groupby("group_id").size()

print("Всего диалогов:", len(biection))
print("Всего групп:", len(full_group_sizes))

too_large = full_group_sizes[
    full_group_sizes > max(N_TRAIN, N_VAL, N_TEST)
]

if not too_large.empty:
    print(
        "Внимание: слишком большие группы не смогут попасть "
        f"в выборку: {len(too_large)} групп, "
        f"{int(too_large.sum())} диалогов."
    )


# ============================================================
# 5. Подбираем целые группы: test=200, val=200, train=933
# ============================================================

rng = np.random.default_rng(SEED)

for attempt in range(100):
    try:
        test_part, remainder = take_exact_groups(
            biection, N_TEST, rng
        )
        val_part, remainder = take_exact_groups(
            remainder, N_VAL, rng
        )
        train_part, _ = take_exact_groups(
            remainder, N_TRAIN, rng
        )
        break

    except ValueError:
        continue

else:
    raise RuntimeError(
        "Не удалось подобрать точные размеры за 100 попыток. "
        "Проверь размеры групп или измени размеры выборок."
    )


# ============================================================
# 6. Создаём итоговые таблицы
# ============================================================

train_part["split"] = "train"
val_part["split"] = "val"
test_part["split"] = "test"

data_little = (
    pd.concat([train_part, val_part, test_part])
    .sort_index()
    .reset_index(drop=True)
)

train = (
    data_little.loc[data_little["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)
val = (
    data_little.loc[data_little["split"] == "val"]
    .copy()
    .reset_index(drop=True)
)
test = (
    data_little.loc[data_little["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 7. Проверяем размеры и отсутствие пересечения групп
# ============================================================

assert len(data_little) == N_TOTAL
assert len(train) == N_TRAIN
assert len(val) == N_VAL
assert len(test) == N_TEST
assert data_little["dialog_id"].is_unique

# Ни одна группа не попала в разные split.
assert (
    data_little.groupby("group_id")["split"].nunique() == 1
).all(), "Есть пересечение групп между выборками"

# Каждая выбранная группа взята целиком из исходных данных.
selected_sizes = data_little.groupby("group_id").size()

assert selected_sizes.eq(
    full_group_sizes.reindex(selected_sizes.index)
).all(), "Есть разрезанные группы"


# ============================================================
# 8. Сохраняем разбиение и выбранные диалоги
# ============================================================

data_little[
    ["dialog_id", "group_id", "split"]
].to_csv(
    "topic_split_small.csv",
    index=False,
)

data_little[
    ["dialog_id", "turns", "group_id", "split"]
].to_json(
    "data_little.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print("\nКоличество диалогов:")
print(
    data_little["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)
print("\nВсего в data_little:", len(data_little))
print("Все проверки пройдены.")

ID в ручном списке: 55
Диалогов в объединённой группе: 55
Всего диалогов: 10000
Всего групп: 9807

Количество диалогов:
split
train    933
val      200
test     200
Name: count, dtype: int64

Всего в data_little: 1333
Все проверки пройдены.


In [57]:
with open('test_split.txt', 'w') as f:
    f.write('\n'.join(x for x in test['dialog_id'].to_list()))

with open('val_split.txt', 'w') as f:
    f.write('\n'.join(x for x in val['dialog_id'].to_list()))

with open('train_split.txt', 'w') as f:
    f.write('\n'.join(x for x in train['dialog_id'].to_list()))

In [259]:
i = 15

train.loc[i]['turns']

[{'role': 'user',
  'text': 'is deforum stable diffusion able to make a video around an object'},
 {'role': 'assistant',
  'text': "Deforum stable diffusion, as a term, does not seem to refer to any specific technology, technique, or process in videography or related fields. If you're asking whether it's possible to make a video around an object, the answer is yes.\n\nYou can create a video around an object using various techniques and camera gear:\n\n1. Circular Tracking Shot: Position the camera on a circular track (e.g., a dolly track) or use a handheld stabilizer to move around an object smoothly.\n\n2. 360-Degree Cameras: Utilize 360-degree or panoramic cameras that capture the entire environment, including an object in the center. These cameras create unique, immersive videos viewable in VR headsets or through interactive video players on social media.\n\n3. Drone Photography: With a drone or quadcopter, you can fly around an object, capturing a video from multiple angles and hei

In [260]:
val.loc[i]['dialog_id']

'd7c3a317263'